# 月×州の売上基準分析

月・州のGMV、注文数、平均注文額を確認する。平均注文額は各期間のGMV合計÷ユニークな注文数で計算する。月別平均注文額の箱ひげ図は注文単位の分布ではない。

共通の集計・作図関数は`../src/`に置き、Notebookには分析の問い、対象データ、結果の読み方を残す。


## データの確認

まず行数、型、欠損、記述統計を確認する。


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").is_dir() and (path / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from src.project_paths import DATA_DIR
from src.sales_metrics import aggregate_sales, add_sales_shares
from src.analysis_plots import (
    plot_monthly_sales_trend, plot_monthly_state_gmv_box,
    plot_state_share_pie, plot_monthly_state_aov_box,
)
df_sales = pd.read_csv(DATA_DIR / "sales_baseline_by_month_state.csv")
df_sales.head()

In [20]:
df_sales.shape

(556, 6)

In [21]:
df_sales.dtypes

purchase_month             str
customer_state             str
order_count              int64
gmv                    float64
total_freight          float64
average_order_value    float64
dtype: object

In [22]:
df_sales.isnull().sum()

purchase_month         0
customer_state         0
order_count            0
gmv                    0
total_freight          0
average_order_value    0
dtype: int64

In [23]:
# 数値列ごとに、件数・平均・ばらつき・分位点・最大最小を確認する
df_sales.describe().T

,count,mean,std,min,25%,50%,75%,max
order_count,556.0,173.521583,431.227619,1.00,13.000000,37.000000,133.500000,3164.000000
gmv,556.0,23779.672860,54532.214576,10.90,2136.665000,6525.375000,18382.627500,424628.360000
total_freight,556.0,3953.733165,7859.215527,8.49,478.602500,1303.410000,3524.605000,56101.490000
average_order_value,556.0,163.542970,72.342723,10.90,126.587305,145.901231,180.252298,683.617273


In [24]:
df_sales.groupby("customer_state")[["order_count", "gmv", "total_freight", "average_order_value"]].describe()

order_count                                                   \
                     count         mean          std   min      25%     50%   
customer_state                                                                
AC                    20.0     4.000000     1.555973   2.0     3.00     4.0   
AL                    21.0    18.904762     8.943739   1.0    12.00    19.0   
AM                    19.0     7.631579     3.876002   1.0     5.00     8.0   
AP                    18.0     3.722222     2.346601   1.0     2.00     3.0   
BA                    21.0   155.047619    73.281291   3.0   103.00   159.0   
CE                    21.0    60.904762    28.569048   6.0    44.00    72.0   
DF                    21.0    99.047619    53.430774   6.0    60.00    94.0   
ES                    21.0    95.000000    46.149756   3.0    79.00   100.0   
GO                    21.0    93.190476    45.129391   7.0    73.00   105.0   
MA                    21.0    34.142857    15.537972   4.0    25.00    38.0   
MG                    21.0   540.666667   255.597209  35.0   356.00   542.0   
MS                    20.0    35.050000    17.706600   1.0    24.00    33.5   
MT                    21.0    42.190476    21.958185   1.0    25.00    39.0   
PA                    21.0    45.047619    18.224918   4.0    36.00    43.0   
PB                    21.0    24.619048    11.762126   1.0    17.00    27.0   
PE                    21.0    75.857143    38.472439   6.0    41.00    83.0   
PI                    21.0    22.666667    10.928556   1.0    14.00    23.0   
PR                    22.0   223.772727   118.487665   1.0   132.50   213.0   
RJ                    21.0   588.095238   267.362844  40.0   399.00   654.0   
RN                    21.0    22.571429    11.373528   4.0    13.00    23.0   
RO                    20.0    12.150000     4.145702   3.0     9.75    12.0   
RR                    17.0     2.411765     1.416811   1.0     1.00     2.0   
RS                    21.0   254.523810   112.016793  17.0   206.00   278.0   
SC                    21.0   168.857143    81.471029   9.0   109.00   174.0   
SE                    21.0    15.952381     6.880961   3.0    12.00    16.0   
SP                    22.0  1840.954545  1062.585458   1.0  1045.50  1694.0   
TO                    20.0    13.700000     5.876805   1.0    11.75    15.5   

                                  gmv                 ... total_freight  \
                    75%     max count           mean  ...           75%   
customer_state                                        ...                 
AC                 5.00     8.0  20.0     796.548500  ...      250.4925   
AL                26.00    36.0  21.0    3755.034286  ...     1050.2300   
AM                 9.00    18.0  19.0    1166.096842  ...      367.8750   
AP                 4.75    11.0  18.0     743.045000  ...      175.2550   
BA               216.00   245.0  21.0   23504.006667  ...     6630.5100   
CE                79.00   103.0  21.0   10464.637143  ...     2797.7800   
DF               146.00   170.0  21.0   14118.971905  ...     3550.1100   
ES               132.00   168.0  21.0   12792.545238  ...     3384.8400   
GO               132.00   153.0  21.0   13468.414286  ...     3391.8400   
MA                46.00    55.0  21.0    5571.875238  ...     2093.6000   
MG               751.00   922.0  21.0   73927.706190  ...    18261.4700   
MS                45.25    70.0  20.0    5771.498500  ...     1254.0950   
MT                58.00    83.0  21.0    7247.220000  ...     2046.0200   
PA                57.00    69.0  21.0    8308.123333  ...     2592.3400   
PB                29.00    51.0  21.0    5361.277143  ...     1318.6300   
PE               102.00   136.0  21.0   11994.737619  ...     3499.9300   
PI                31.00    47.0  21.0    4034.333333  ...     1448.1400   
PR               323.75   381.0  22.0   30275.614091  ...     7987.4025   
RJ               807.00  1012.0  21.0   83792.910952  ...    19710.03

In [25]:
# 各数値列で最大値となる月・州の組み合わせを確認する

for column in ["order_count", "gmv", "total_freight", "average_order_value"]:
    max_index = df_sales[column].idxmax()
    print(f"\n{column}")
    display(df_sales.loc[max_index, ["purchase_month", "customer_state", column]])



order_count


purchase_month    2018-08
customer_state         SP
order_count          3164
Name: 554, dtype: object


gmv


purchase_month      2018-05
customer_state           SP
gmv               424628.36
Name: 474, dtype: object


total_freight


purchase_month     2018-08
customer_state          SP
total_freight     56101.49
Name: 554, dtype: object


average_order_value


purchase_month            2017-02
customer_state                 MS
average_order_value    683.617273
Name: 58, dtype: object

In [26]:
# 平均注文額が最大だった月・州の行を、注文数とGMVも含めて確認する
max_aov_index = df_sales["average_order_value"].idxmax()

display(
    df_sales.loc[
        max_aov_index,
        [
            "purchase_month",
            "customer_state",
            "order_count",
            "gmv",
            "total_freight",
            "average_order_value",
        ],
    ]
)

# 全体の加重平均注文額を計算する
weighted_average_order_value = (
    df_sales["gmv"].sum() / df_sales["order_count"].sum()
)

print("全体の加重平均注文額:", round(weighted_average_order_value, 2))

purchase_month            2017-02
customer_state                 MS
order_count                    11
gmv                       7519.79
total_freight              391.15
average_order_value    683.617273
Name: 58, dtype: object

全体の加重平均注文額: 137.04


## 月別・州別の基準値

GMVと注文数を合計してからAOVを再計算する。


In [ ]:
# 月のGMV・注文数を合計してからAOVを再計算する。
df_monthly = aggregate_sales(df_sales, "purchase_month")
df_monthly = df_monthly.sort_values("purchase_month").reset_index(drop=True)
display(df_monthly)

In [ ]:
# GMVの増減を注文数と一緒に読む。
fig = plot_monthly_sales_trend(df_monthly)

In [ ]:
# 州別のAOVも、月別AOVの単純平均ではなく合計GMV÷合計注文数。
df_state = aggregate_sales(df_sales, "customer_state")
df_state = df_state.sort_values("gmv", ascending=False).reset_index(drop=True)
display(df_state)

SP州が一番GMVが大きいが、全体の加重平均額137.04（データ上の金額単位）よりは低い。

In [ ]:
# 州全体の注文数・GMVを分母にしてシェアを付ける。
df_state = add_sales_shares(df_state)
display(df_state[[
    "customer_state", "order_count", "order_share_pct", "gmv",
    "gmv_share_pct", "average_order_value",
]].head(10).round(2))

In [31]:
# GMVの大きい上位3州を取り出す
top_3_states = df_state.head(3)

# 上位3州が全体に占める割合を合計する
top_3_order_share = top_3_states["order_share_pct"].sum()
top_3_gmv_share = top_3_states["gmv_share_pct"].sum()

print(f"上位3州の注文数シェア: {top_3_order_share:.2f}%")
print(f"上位3州のGMVシェア: {top_3_gmv_share:.2f}%")

上位3州の注文数シェア: 66.55%
上位3州のGMVシェア: 63.38%


## 州別の分布と構成比

月×州の分布と州全体のシェアは集計単位が異なる。


In [ ]:
# 月×州GMVの分布。州合計GMVが大きい順に並べる。
top5_gmv_states = ["SP", "RJ", "MG", "RS", "PR"]
top5_avg_states = ["AP", "PB", "AC", "AL", "PA"]
fig = plot_monthly_state_gmv_box(df_sales, top5_gmv_states, top5_avg_states)

In [ ]:
# GMVに占める選択州とその他の割合。
fig = plot_state_share_pie(
    df_state, "gmv_share_pct", top5_gmv_states, top5_avg_states,
    "GMV Share: Top GMV and AOV States",
)

In [ ]:
# 分母をGMVから注文数に変えて、同じ州を比較する。
fig = plot_state_share_pie(
    df_state, "order_share_pct", top5_gmv_states, top5_avg_states,
    "Order Share: Top Order and AOV States",
)

In [ ]:
# 月×州の平均注文額の分布。注文1件ごとの分布とは異なる。
fig = plot_monthly_state_aov_box(
    df_sales, "mean", top5_gmv_states, top5_avg_states
)

In [ ]:
# 並び順を「各州の月別AOVの中央値」に変える。
fig = plot_monthly_state_aov_box(
    df_sales, "median", top5_gmv_states, top5_avg_states
)

### 図の集計単位と使い方

州別GMVの箱ひげ図は「月×州のGMV」、平均注文額の箱ひげ図は「月×州の平均注文額」の分布を表す。箱の中央値は注文1件ごとの金額の中央値ではない。州の全注文に対する平均注文額を比べる場合は、上で求めた`df_state["average_order_value"]`（州のGMV合計÷州の注文数合計）を使う。注文額そのものの中央値には注文単位のデータが必要。
